# Séance 3 · Exercices — SQL et Git · ⭐

**Niveau : ⭐ Débutant**

**Niveau de la séance : ⭐ Débutant**

Comment travailler :
1. Lis l'énoncé, puis code dans la cellule « À toi » (les variables à remplir sont déjà nommées).
2. Lance la cellule de vérification : ✅ c'est bon, ❌ on réessaie (l'indice est là pour ça).
3. Ouvre la solution **seulement après avoir essayé**.

Ce notebook tourne dans **Google Colab** : rien à installer. Pour exécuter une cellule : clique dedans puis `Maj + Entrée`.


## Préparation

Comme dans la leçon : le dataset Pokémon avec des colonnes en français, rangé dans une base **SQLite en mémoire** (tables `pokemon` et `types`), la fonction `sql(requete)`, et `valeurs(tableau)` pour comparer un résultat SQL à un résultat pandas.

In [ ]:
import sqlite3
import pandas as pd

url = "https://gist.githubusercontent.com/armgilles/194bcff35001e7eb53a2a8b441e8b2c6/raw/92200bc0a673d5ce2110aaad4544ed6c4010f687/pokemon.csv"
try:
    df = pd.read_csv(url)
    print("Dataset Pokémon chargé :", len(df), "lignes")
except Exception as e:
    print("Pas de réseau ? On utilise une mini-version de secours (les vérifications supposent le dataset complet).", e)
    df = pd.DataFrame({
        "#": [1, 4, 7, 25, 94, 130, 143, 150], "Name": ["Bulbasaur", "Charmander", "Squirtle", "Pikachu", "Gengar", "Gyarados", "Snorlax", "Mewtwo"],
        "Type 1": ["Grass", "Fire", "Water", "Electric", "Ghost", "Water", "Normal", "Psychic"], "Type 2": ["Poison", None, None, None, "Poison", "Flying", None, None],
        "Total": [318, 309, 314, 320, 500, 540, 540, 680], "HP": [45, 39, 44, 35, 60, 95, 160, 106], "Attack": [49, 52, 48, 55, 65, 125, 110, 110],
        "Defense": [49, 43, 65, 40, 60, 79, 65, 90], "Sp. Atk": [65, 60, 50, 50, 130, 60, 65, 154], "Sp. Def": [65, 50, 64, 50, 75, 100, 110, 90],
        "Speed": [45, 65, 43, 90, 110, 81, 30, 130], "Generation": [1] * 8, "Legendary": [False] * 7 + [True],
    })

# Colonnes en français, comme dans la leçon
df = df.rename(columns={
    "#": "numero", "Name": "nom", "Type 1": "type1", "Type 2": "type2", "Total": "total", "HP": "pv", "Attack": "attaque",
    "Defense": "defense", "Sp. Atk": "attaque_spe", "Sp. Def": "defense_spe", "Speed": "vitesse", "Generation": "generation", "Legendary": "legendaire",
})
df["legendaire"] = df["legendaire"].astype(int)     # 1 = légendaire, 0 = non

# La base SQLite en mémoire, avec les tables pokemon et types
con = sqlite3.connect(":memory:")
df.to_sql("pokemon", con, index=False)

types = pd.DataFrame({
    "type": ["Fire", "Water", "Grass", "Electric", "Ground", "Rock", "Ice", "Psychic", "Fighting", "Flying"],
    "fort_contre": ["Grass", "Fire", "Water", "Water", "Electric", "Fire", "Grass", "Fighting", "Normal", "Grass"],
    "faible_contre": ["Water", "Electric", "Fire", "Ground", "Water", "Water", "Fire", "Bug", "Psychic", "Electric"],
})
types.to_sql("types", con, index=False)

def sql(requete):
    """Exécute une requête SQL et renvoie le résultat sous forme de tableau pandas."""
    return pd.read_sql_query(requete, con)

def valeurs(tableau):
    """Les valeurs d'un tableau (sans les noms de colonnes), arrondies à 2 décimales : pour comparer SQL et pandas."""
    tableau = pd.DataFrame(tableau).reset_index(drop=True)
    return [tuple(round(v, 2) if isinstance(v, float) else v for v in ligne) for ligne in tableau.values.tolist()]

print("Tables :", sql("SELECT name FROM sqlite_master")["name"].tolist())

resultats = {}

def verifier(nom, condition):
    """Affiche ✅ ou ❌ sans jamais lever d'exception (condition = un booléen)."""
    try:
        ok = bool(condition() if callable(condition) else condition)
    except Exception:
        ok = False
    resultats[nom] = ok
    print(("✅ " if ok else "❌ ") + nom + ("" if ok else " — pas encore, relis l'énoncé ou ouvre l'indice."))

print("Prêt !")

## Exercice 1 ⭐ · SELECT et LIMIT

Écris la requête qui affiche les colonnes `nom`, `type1` et `pv` des **5 premiers** Pokémon (dans cet ordre de colonnes).

<details><summary>Indice</summary>

`SELECT nom, type1, pv FROM pokemon LIMIT 5`.

</details>

In [ ]:
# À toi : modifie la requête
requete = """
SELECT nom FROM pokemon LIMIT 1
"""
resultat = sql(requete)
resultat

In [ ]:
verifier("Exercice 1", list(resultat.columns) == ["nom", "type1", "pv"] and len(resultat) == 5)

<details><summary>Solution</summary>

```python
requete = """
SELECT nom, type1, pv
FROM pokemon
LIMIT 5
"""
resultat = sql(requete)
resultat
```

</details>

## Exercice 2 ⭐ · WHERE et COUNT

Combien de Pokémon sont de type `Electric` (colonne `type1`) ? La requête doit renvoyer **une seule ligne** avec le nombre, nommé `nb`. Attendu : `44`.

<details><summary>Indice</summary>

`SELECT COUNT(*) AS nb FROM pokemon WHERE type1 = 'Electric'` (texte entre guillemets simples).

</details>

In [ ]:
# À toi
requete = """
SELECT COUNT(*) AS nb FROM pokemon
"""
resultat = sql(requete)
resultat

In [ ]:
verifier("Exercice 2", "nb" in resultat.columns and resultat["nb"].iloc[0] == 44)

<details><summary>Solution</summary>

```python
requete = """
SELECT COUNT(*) AS nb
FROM pokemon
WHERE type1 = 'Electric'
"""
resultat = sql(requete)
resultat
```

</details>

## Exercice 3 ⭐ · Deux conditions

Le nom et la vitesse des Pokémon de la **génération 1** dont la vitesse est **supérieure ou égale à 120**. Attendu : 13 lignes, dont `Electrode`.

<details><summary>Indice</summary>

`WHERE generation = 1 AND vitesse >= 120`.

</details>

In [ ]:
# À toi
requete = """
SELECT nom, vitesse FROM pokemon LIMIT 3
"""
resultat = sql(requete)
resultat

In [ ]:
verifier("Exercice 3", len(resultat) == 13 and "Electrode" in resultat["nom"].tolist() and resultat["vitesse"].min() >= 120)

<details><summary>Solution</summary>

```python
requete = """
SELECT nom, vitesse
FROM pokemon
WHERE generation = 1 AND vitesse >= 120
"""
resultat = sql(requete)
resultat
```

</details>

## Exercice 4 ⭐ · ORDER BY : le podium de la défense

Le nom et la défense des **3** Pokémon les mieux défendus (attendu : trois Pokémon à 230 de défense, dont `Shuckle`).

<details><summary>Indice</summary>

`ORDER BY defense DESC LIMIT 3`.

</details>

In [ ]:
# À toi
requete = """
SELECT nom, defense FROM pokemon ORDER BY defense LIMIT 3
"""
resultat = sql(requete)
resultat

In [ ]:
verifier("Exercice 4", set(resultat["nom"]) == {"SteelixMega Steelix", "Shuckle", "AggronMega Aggron"})

<details><summary>Solution</summary>

```python
requete = """
SELECT nom, defense
FROM pokemon
ORDER BY defense DESC
LIMIT 3
"""
resultat = sql(requete)
resultat
```

</details>

## Exercice 5 ⭐⭐ · COUNT et AVG ensemble

En une seule requête : le nombre de Pokémon légendaires (`nb`) et leur attaque moyenne arrondie à 1 décimale (`attaque_moyenne`).
Attendu : `65` et `116.7`.

<details><summary>Indice</summary>

`SELECT COUNT(*) AS nb, ROUND(AVG(attaque), 1) AS attaque_moyenne FROM pokemon WHERE legendaire = 1`.

</details>

In [ ]:
# À toi
requete = """
SELECT COUNT(*) AS nb FROM pokemon
"""
resultat = sql(requete)
resultat

In [ ]:
verifier("Exercice 5", list(resultat.columns) == ["nb", "attaque_moyenne"] and resultat["nb"].iloc[0] == 65 and resultat["attaque_moyenne"].iloc[0] == 116.7)

<details><summary>Solution</summary>

```python
requete = """
SELECT COUNT(*) AS nb, ROUND(AVG(attaque), 1) AS attaque_moyenne
FROM pokemon
WHERE legendaire = 1
"""
resultat = sql(requete)
resultat
```

</details>

## Exercice 6 ⭐⭐ · GROUP BY : combien par type ?

Le nombre de Pokémon par `type1` (colonne `nb`), du type le plus fréquent au plus rare. Attendu : 18 lignes, `Water` en tête avec 112.

<details><summary>Indice</summary>

`GROUP BY type1 ORDER BY nb DESC`.

</details>

In [ ]:
# À toi
requete = """
SELECT type1, COUNT(*) AS nb FROM pokemon GROUP BY type1
"""
resultat = sql(requete)
resultat

In [ ]:
verifier("Exercice 6", len(resultat) == 18 and resultat["type1"].iloc[0] == "Water" and resultat["nb"].iloc[0] == 112)

<details><summary>Solution</summary>

```python
requete = """
SELECT type1, COUNT(*) AS nb
FROM pokemon
GROUP BY type1
ORDER BY nb DESC
"""
resultat = sql(requete)
resultat
```

</details>

## Exercice 7 ⭐⭐ · Même question, SQL et pandas

Les PV moyens par génération, arrondis à 1 décimale, de la génération la plus résistante à la moins résistante. Une fois en SQL (`en_sql`), une fois en pandas (`en_pandas`).
La vérification compare les valeurs des deux tableaux avec `valeurs(...)`.

Attendu : la génération 4 en tête avec `73.1`, la génération 1 en dernier avec `65.8`.

<details><summary>Indice</summary>

pandas : `df.groupby("generation")["pv"].mean().round(1).sort_values(ascending=False).reset_index()`. SQL : `ROUND(AVG(pv), 1) AS pv_moyen ... GROUP BY generation ORDER BY pv_moyen DESC`.

</details>

In [ ]:
# À toi
en_sql = sql("SELECT generation, ROUND(AVG(pv), 1) AS pv_moyen FROM pokemon GROUP BY generation")
en_pandas = df.groupby("generation")["pv"].mean().round(1).reset_index()

print(en_sql)
print(en_pandas)

In [ ]:
verifier("Exercice 7", valeurs(en_sql) == valeurs(en_pandas) == [(4, 73.1), (5, 71.8), (2, 71.2), (6, 68.3), (3, 66.5), (1, 65.8)])

<details><summary>Solution</summary>

```python
en_sql = sql("""
SELECT generation, ROUND(AVG(pv), 1) AS pv_moyen
FROM pokemon
GROUP BY generation
ORDER BY pv_moyen DESC
""")
en_pandas = df.groupby("generation")["pv"].mean().round(1).sort_values(ascending=False).reset_index()

print(en_sql)
print(en_pandas)
```

</details>

## Exercice 8 ⭐⭐ · Quelle commande git ?

Pour chaque situation, écris la commande git qui correspond (juste les deux premiers mots, par exemple `"git add"`).
Rappel de la leçon : `add` (choisir), `commit` (sauvegarder avec un message), `push` (envoyer), `pull` (récupérer).

<details><summary>Indice</summary>

Point de sauvegarde = commit ; envoyer vers GitHub = push ; récupérer depuis GitHub = pull ; préparer les fichiers = add.

</details>

In [ ]:
commandes = {
    "Je choisis les fichiers modifiés à mettre dans la prochaine sauvegarde": "?",
    "Je crée un point de sauvegarde avec un message qui dit pourquoi": "?",
    "J'envoie mes sauvegardes sur GitHub pour les montrer": "?",
    "Sur un autre ordinateur, je récupère ce qui a changé sur GitHub": "?",
}

# À toi : remplace les "?" puis exécute
for situation, commande in commandes.items():
    print(f"{commande:12s} <- {situation}")

In [ ]:
verifier("Exercice 8", [" ".join(c.strip().lower().split()[:2]) for c in commandes.values()] == ["git add", "git commit", "git push", "git pull"])

<details><summary>Solution</summary>

```python
commandes = {
    "Je choisis les fichiers modifiés à mettre dans la prochaine sauvegarde": "git add",
    "Je crée un point de sauvegarde avec un message qui dit pourquoi": "git commit",
    "J'envoie mes sauvegardes sur GitHub pour les montrer": "git push",
    "Sur un autre ordinateur, je récupère ce qui a changé sur GitHub": "git pull",
}

for situation, commande in commandes.items():
    print(f"{commande:12s} <- {situation}")
```

</details>

## Exercice 9 ⭐⭐⭐ · JOIN : forts contre l'herbe

Combien de Pokémon ont un type **fort contre l'herbe** (`fort_contre = 'Grass'` dans la table `types`) ? Résultat dans une colonne `nb`. Attendu : `80`.

<details><summary>Indice</summary>

`FROM pokemon AS p JOIN types AS t ON p.type1 = t.type WHERE t.fort_contre = 'Grass'`.

</details>

In [ ]:
# À toi
requete = """
SELECT COUNT(*) AS nb
FROM pokemon AS p
JOIN types AS t ON p.type1 = t.type
"""
resultat = sql(requete)
resultat

In [ ]:
verifier("Exercice 9", "nb" in resultat.columns and resultat["nb"].iloc[0] == 80)

<details><summary>Solution</summary>

```python
requete = """
SELECT COUNT(*) AS nb
FROM pokemon AS p
JOIN types AS t ON p.type1 = t.type
WHERE t.fort_contre = 'Grass'
"""
resultat = sql(requete)
resultat
```

</details>

## Exercice 10 ⭐⭐⭐ · LEFT JOIN : les oubliés

Un `JOIN` simple ignore les Pokémon dont le type n'est pas dans `types`. Avec `LEFT JOIN`, on les garde : leurs colonnes venant de `types` sont vides (`NULL`).
Compte les Pokémon **sans correspondance** dans `types` grâce à `WHERE t.type IS NULL`. Colonne `nb`, attendu : `334`.

<details><summary>Indice</summary>

`FROM pokemon AS p LEFT JOIN types AS t ON p.type1 = t.type WHERE t.type IS NULL`.

</details>

In [ ]:
# À toi
requete = """
SELECT COUNT(*) AS nb
FROM pokemon AS p
LEFT JOIN types AS t ON p.type1 = t.type
"""
resultat = sql(requete)
resultat

In [ ]:
verifier("Exercice 10", "nb" in resultat.columns and resultat["nb"].iloc[0] == 334)

<details><summary>Solution</summary>

```python
requete = """
SELECT COUNT(*) AS nb
FROM pokemon AS p
LEFT JOIN types AS t ON p.type1 = t.type
WHERE t.type IS NULL
"""
resultat = sql(requete)
resultat
```

</details>

## Exercice 11 ⭐⭐⭐ · Git : l'ordre et le bon message

1. `ordre` : les trois commandes, dans l'ordre, pour envoyer une modification sur GitHub (deux mots chacune, par exemple `"git add"`).
2. `bons_messages` : parmi les 6 messages de commit proposés, garde ceux qui disent **l'intention** (ce qui a été fait et pourquoi), pas ceux qui ne disent rien.

<details><summary>Indice</summary>

Un bon message commence par un verbe et décrit le changement : « Ajoute... », « Corrige... ». « modif », « test », « v2 » ne disent rien à quelqu'un qui relit l'historique.

</details>

In [ ]:
messages = [
    "modif",
    "Ajoute le graphique vitesse par génération",
    "test",
    "Corrige le calcul de la moyenne des PV",
    "fichier final v2",
    "Ajoute un README qui explique le projet",
]

# À toi
ordre = ["?", "?", "?"]
bons_messages = []

print(ordre)
print(bons_messages)

In [ ]:
verifier("Exercice 11", [" ".join(c.strip().lower().split()[:2]) for c in ordre] == ["git add", "git commit", "git push"]
         and set(bons_messages) == {"Ajoute le graphique vitesse par génération", "Corrige le calcul de la moyenne des PV", "Ajoute un README qui explique le projet"})

<details><summary>Solution</summary>

```python
messages = [
    "modif",
    "Ajoute le graphique vitesse par génération",
    "test",
    "Corrige le calcul de la moyenne des PV",
    "fichier final v2",
    "Ajoute un README qui explique le projet",
]

ordre = ["git add", "git commit", "git push"]
bons_messages = [messages[1], messages[3], messages[5]]

print(ordre)
print(bons_messages)
```

</details>

## Exercice 12 ⭐⭐⭐ · Défi : JOIN + GROUP BY, en SQL et en pandas

Pour chaque faiblesse (`faible_contre` de la table `types`), la vitesse moyenne des Pokémon concernés, arrondie à 1 décimale, les **3** plus rapides en premier.
Une fois en SQL (`en_sql`), une fois en pandas (`en_pandas`, avec `merge` puis `groupby`).

Attendu : `Ground` (84.5), `Bug` (81.5), `Electric` (67.2).

<details><summary>Indice</summary>

pandas : `df.merge(types, left_on="type1", right_on="type").groupby("faible_contre")["vitesse"].mean().round(1).sort_values(ascending=False).head(3).reset_index()`. SQL : `JOIN ... GROUP BY t.faible_contre ORDER BY vitesse_moyenne DESC LIMIT 3`.

</details>

In [ ]:
# À toi
en_sql = sql("SELECT t.faible_contre, COUNT(*) AS nb FROM pokemon AS p JOIN types AS t ON p.type1 = t.type GROUP BY t.faible_contre")
en_pandas = df.merge(types, left_on="type1", right_on="type").groupby("faible_contre").size().reset_index(name="nb")

print(en_sql)
print(en_pandas)

In [ ]:
verifier("Exercice 12", valeurs(en_sql) == valeurs(en_pandas) == [("Ground", 84.5), ("Bug", 81.5), ("Electric", 67.2)])

<details><summary>Solution</summary>

```python
en_sql = sql("""
SELECT t.faible_contre, ROUND(AVG(p.vitesse), 1) AS vitesse_moyenne
FROM pokemon AS p
JOIN types AS t ON p.type1 = t.type
GROUP BY t.faible_contre
ORDER BY vitesse_moyenne DESC
LIMIT 3
""")
en_pandas = (df.merge(types, left_on="type1", right_on="type")
               .groupby("faible_contre")["vitesse"].mean().round(1)
               .sort_values(ascending=False).head(3).reset_index())

print(en_sql)
print(en_pandas)
```

</details>

## Bilan

La cellule ci-dessous compte tes ✅. Relance-la après chaque exercice réussi.

In [ ]:
reussis = sum(1 for ok in resultats.values() if ok)
print(f"{reussis} exercice(s) réussi(s) sur {len(resultats)}")
for nom, ok in resultats.items():
    print("  ", "✅" if ok else "❌", nom)
